# TempFlow: forecasting land-surface temperature

A stacked bidirectional LSTM (BiLSTM 128, LSTM 64, LSTM 32, dense) that forecasts daytime MODIS LST from a 30-day context of 8 inputs per day: latitude, longitude, year, month and day of year as sine and cosine, and the LST itself, with gaps of up to 7 days filled by linear interpolation. The data are split chronologically at 7 June 2018 and the scalers are fitted on the training period only. Training: MSE loss, AdamW (learning rate 1e-3, weight decay 4e-3), batch 512, early stopping on 15 % of the training sequences.

Baselines on the same targets: persistence (the last value of the context), the seasonal cycle of each pixel (annual and semi-annual harmonics fitted on the training period) and the day-of-year mean over the grid. One model is trained for each lead time from 1 to 4 days.

`SITE = 'lake'` uses the 8 x 8 grid at the lake (notebook 04); `SITE = 'original_points'` uses the 64 points of the GLOFNet LST data about 14 km north-west of the lake (`data/lst_original_points.csv.gz`).

**Output** (`TempFlow_Shisper` or `TempFlow_original`): test scores, scores by lead time, three figures and the next-day forecast for every pixel-day used by FusionFlow. GPU runtime recommended; saved models are reused on a rerun.

## Setup

In [ ]:
import os, json
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from matplotlib import dates as mdates, rcParams

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
REPO = 'https://raw.githubusercontent.com/ZuhaFatima96/IceWatch-A-Multimodal-Deep-Learning-Framework-for-Early-Warning-of-GLOFs/main'
DATA = '../data' if os.path.isdir('../data') else f'{REPO}/data'
SITE = 'lake'                                       # 'lake' or 'original_points'
if SITE == 'lake':
    LST, OUT = f'{DRIVE}/Shisper_LST_MODIS/lst_lake.csv', f'{DRIVE}/TempFlow_Shisper'
else:
    LST, OUT = f'{DATA}/lst_original_points.csv.gz', f'{DRIVE}/TempFlow_original'
os.makedirs(OUT, exist_ok=True)

SEQ = 30
FEATS = ['Latitude', 'Longitude', 'year', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos', 'ctx']
CUT = pd.Timestamp('2018-06-07')                    # chronological split
HORIZONS = [1, 2, 3, 4]
EPOCHS = 100                                        # upper limit; early stopping decides
SEED = 42
print('GPU:', tf.config.list_physical_devices('GPU'), '| data:', LST)

## Data preparation

In [ ]:
df = pd.read_csv(LST, usecols=['Latitude', 'Longitude', 'Date', 'MOD11A1_061_LST_Day_1km'])
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
df['pixel'] = df['Latitude'].round(4).astype(str) + '_' + df['Longitude'].round(4).astype(str)
df['lst'] = pd.to_numeric(df['MOD11A1_061_LST_Day_1km'], errors='coerce')
df['real'] = df['lst'].where(df['lst'] > 0)          # 0 = no retrieval (cloud)

# daily series per pixel; ctx = LST with gaps of up to 7 days filled by linear interpolation
parts = []
for pid, p in df.groupby('pixel'):
    p = p.sort_values('Date').drop_duplicates('Date').set_index('Date')
    p = p.reindex(pd.date_range(p.index.min(), p.index.max(), freq='D'))
    p['pixel'] = pid
    p['Latitude'] = p['Latitude'].ffill().bfill()
    p['Longitude'] = p['Longitude'].ffill().bfill()
    p['ctx'] = p['real'].interpolate(method='linear', limit=7, limit_area='inside')
    parts.append(p.reset_index().rename(columns={'index': 'Date'}))
df = pd.concat(parts, ignore_index=True)
df['year'] = df['Date'].dt.year
m_, doy_ = df['Date'].dt.month, df['Date'].dt.dayofyear
df['month_sin'] = np.sin(2 * np.pi * m_ / 12);       df['month_cos'] = np.cos(2 * np.pi * m_ / 12)
df['doy_sin'] = np.sin(2 * np.pi * doy_ / 365.25);   df['doy_cos'] = np.cos(2 * np.pi * doy_ / 365.25)

TR = df[df['Date'] < CUT].copy(); TE = df[df['Date'] >= CUT].copy()
print('pixels:', df.pixel.nunique(), '| days:', df.Date.min().date(), 'to', df.Date.max().date(),
      f"| cloud-free: {df.real.notna().mean():.1%}")

sx = StandardScaler().fit(TR.loc[TR['ctx'].notna(), FEATS])
sy = StandardScaler().fit(TR.loc[TR['real'].notna(), ['real']])
YMEAN, YSCALE = float(sy.mean_[0]), float(sy.scale_[0])

def make(d, H):
    # complete 30-day contexts and the cloud-free LST H days after the last context day
    X, y, pers, pix, dts = [], [], [], [], []
    for pid, p in d.groupby('pixel'):
        p = p.sort_values('Date'); n = len(p)
        if n < SEQ + H:
            continue
        ok = p['ctx'].notna().values; rt = p['real'].notna().values
        Fs = np.full((n, len(FEATS)), np.nan, dtype='float32')
        Fs[ok] = sx.transform(p.loc[ok, FEATS])
        c = np.r_[0, np.cumsum(ok)]
        win_ok = (c[SEQ:] - c[:-SEQ]) == SEQ
        i = np.nonzero(win_ok[:n - SEQ - H + 1] & rt[SEQ - 1 + H:])[0]
        if not len(i):
            continue
        W = np.lib.stride_tricks.sliding_window_view(Fs, SEQ, axis=0)
        j = i + SEQ - 1 + H
        X.append(W[i].transpose(0, 2, 1)); y.append(p['real'].values[j]); pers.append(p['ctx'].values[i + SEQ - 1])
        pix.append(np.full(len(i), pid, dtype=object)); dts.append(p['Date'].values[j])
    return dict(X=np.concatenate(X).astype('float32'), y=np.concatenate(y), pers=np.concatenate(pers),
                pix=np.concatenate(pix), date=pd.to_datetime(np.concatenate(dts)))

# climatologies from the training period
clim_grid = (TR.loc[TR['real'].notna()].assign(doy=TR.loc[TR['real'].notna(), 'Date'].dt.dayofyear)
               .groupby('doy')['real'].mean())
def harm(dates):
    w = 2 * np.pi * pd.DatetimeIndex(dates).dayofyear.values / 365.25
    return np.column_stack([np.ones(len(w)), np.sin(w), np.cos(w), np.sin(2 * w), np.cos(2 * w)])
_ok = TR['real'].notna()
_all = np.linalg.lstsq(harm(TR.loc[_ok, 'Date']), TR.loc[_ok, 'real'].values, rcond=None)[0]
HCOEF = {pid: np.linalg.lstsq(harm(p['Date']), p['real'].values, rcond=None)[0] if len(p) >= 50 else _all
         for pid, p in TR[_ok].groupby('pixel')}
def clim_pixel(pix, dates):
    return (harm(dates) * np.vstack([HCOEF.get(q, _all) for q in pix])).sum(1)
def clim_doy(dates):
    return pd.Series(pd.DatetimeIndex(dates).dayofyear).map(clim_grid).values

SEQS = {H: dict(tr=make(TR, H), te=make(TE, H)) for H in HORIZONS}
for H in HORIZONS:
    print(f"lead {H} d: training sequences {len(SEQS[H]['tr']['y']):,} | test targets {len(SEQS[H]['te']['y']):,}")

## Next-day forecast

In [ ]:
rcParams.update({'font.family': 'serif', 'font.size': 10, 'axes.linewidth': 0.8, 'savefig.dpi': 300})

def build():
    m = tf.keras.Sequential([
        tf.keras.layers.Input((SEQ, len(FEATS))),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)), tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(64, return_sequences=True), tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(32), tf.keras.layers.Dropout(0.3), tf.keras.layers.Dense(1)])
    m.compile(tf.keras.optimizers.AdamW(1e-3, weight_decay=4e-3), loss='mse', metrics=['mae'])
    return m

def fit_or_load(H):
    path = f'{OUT}/tempflow_lead{H}.weights.h5'
    tf.keras.utils.set_random_seed(SEED)
    m = build()
    if os.path.exists(path):
        m.load_weights(path)
        return m
    tr = SEQS[H]['tr']
    m.fit(tr['X'], ((tr['y'] - YMEAN) / YSCALE).astype('float32'), validation_split=0.15, epochs=EPOCHS, batch_size=512,
          verbose=2, callbacks=[tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
                                tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)])
    m.save_weights(path)
    return m

def evaluate(H, m):
    te = SEQS[H]['te']
    p = m.predict(te['X'], batch_size=512, verbose=0).ravel() * YSCALE + YMEAN
    y, cp, cg = te['y'], clim_pixel(te['pix'], te['date']), clim_doy(te['date'])
    ok = ~np.isnan(cg)
    return p, dict(lead_days=H, n=int(len(y)), tempflow=round(mean_absolute_error(y, p), 3),
                   persistence=round(mean_absolute_error(y, te['pers']), 3),
                   climatology_pixel=round(mean_absolute_error(y, cp), 3),
                   climatology_grid=round(mean_absolute_error(y[ok], cg[ok]), 3))

MODELS, LEAD = {1: fit_or_load(1)}, []
pred, row = evaluate(1, MODELS[1])
LEAD.append(row)
te1 = SEQS[1]['te']; yte = te1['y']
rmse = float(np.sqrt(np.mean((yte - pred) ** 2))); r2 = float(r2_score(yte, pred))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(yte, pred, s=6, alpha=0.25, color='#3a6ea5', edgecolor='none')
lo, hi = min(yte.min(), pred.min()), max(yte.max(), pred.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel('Observed LST (K)'); ax.set_ylabel('Predicted LST (K)'); ax.set_aspect('equal')
ax.text(0.05, 0.95, f"MAE = {row['tempflow']:.2f} K\nRMSE = {rmse:.2f} K\n$R^2$ = {r2:.3f}\nn = {len(yte)}",
        transform=ax.transAxes, va='top', bbox=dict(boxstyle='round', fc='white', ec='0.7'))
fig.tight_layout(); fig.savefig(f'{OUT}/fig_tempflow_scatter.png'); fig.savefig(f'{OUT}/fig_tempflow_scatter.pdf'); plt.show()

# every day of 2022 with at least one cloud-free target, averaged over the grid
D22 = pd.DataFrame({'date': te1['date'], 'obs': yte, 'pred': pred})
D22 = D22[D22.date.dt.year == 2022]
g = (D22.groupby(D22.date.dt.normalize()).agg(obs=('obs', 'mean'), pred=('pred', 'mean'), n=('obs', 'size'))
       .reset_index().rename(columns={'date': 'day'}).sort_values('day').reset_index(drop=True))
mae22 = float(mean_absolute_error(D22.obs, D22.pred)) if len(D22) else float('nan')
xs, yo, yp = [], [], []
for i in range(len(g)):
    if i > 0 and (g.day[i] - g.day[i - 1]).days > 12:            # no line across gaps longer than 12 days
        xs.append(g.day[i - 1] + pd.Timedelta(days=1)); yo.append(np.nan); yp.append(np.nan)
    xs.append(g.day[i]); yo.append(g.obs[i]); yp.append(g.pred[i])
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.axvspan(pd.Timestamp('2022-05-07'), pd.Timestamp('2022-06-06'), color='#f2c14e', alpha=0.25, zorder=0)
ax.axvline(pd.Timestamp('2022-05-07'), color='0.4', ls='--', lw=1, zorder=1)
ax.plot(xs, yo, '-', color='#2e6b8a', lw=1.4, zorder=2); ax.plot(xs, yp, '-', color='#111111', lw=1.4, zorder=2)
ax.plot(g.day, g.obs, 'o', color='#2e6b8a', ms=4, label='Observed', zorder=3)
ax.plot(g.day, g.pred, 'o', color='#111111', ms=3.5, label='TempFlow forecast', zorder=3)
ax.text(pd.Timestamp('2022-05-09'), 0.97, 'outburst\n7 May', transform=ax.get_xaxis_transform(), fontsize=7.5,
        va='top', color='0.35')
ax.set_ylabel('Land-surface temp. (K)'); ax.set_xlabel('2022')
ax.xaxis.set_major_locator(mdates.MonthLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.legend(loc='lower center', frameon=True, framealpha=0.95, edgecolor='0.8', fontsize=8.5, ncol=2)
ax.text(0.015, 0.96, f'2022 daily MAE = {mae22:.2f} K', transform=ax.transAxes, va='top', fontsize=8.5,
        bbox=dict(boxstyle='round', fc='white', ec='0.8'))
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
ax.grid(True, axis='y', lw=0.4, color='0.9'); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(f'{OUT}/fig_tempflow_2022.png'); fig.savefig(f'{OUT}/fig_tempflow_2022.pdf'); plt.show()

np.savez(f'{OUT}/tempflow_test_predictions.npz', pred=pred, obs=yte, persistence=te1['pers'],
         dates=te1['date'].astype(str).values, pixels=te1['pix'].astype(str))
METRICS = dict(n=row['n'], mae=row['tempflow'], rmse=round(rmse, 3), r2=round(r2, 4), persistence=row['persistence'],
               climatology_pixel=row['climatology_pixel'], climatology_grid=row['climatology_grid'],
               mae_2022=round(mae22, 3), days_2022=int(len(g)), samples_2022=int(len(D22)),
               training_sequences=int(len(SEQS[1]['tr']['y'])), test_period=f'{CUT.date()} to {df.Date.max().date()}')
json.dump(METRICS, open(f'{OUT}/tempflow_metrics.json', 'w'), indent=2)
print(METRICS)

## Lead time

In [ ]:
for H in HORIZONS:
    if H != 1:
        MODELS[H] = fit_or_load(H)
        LEAD.append(evaluate(H, MODELS[H])[1])
LT = pd.DataFrame(LEAD).sort_values('lead_days').reset_index(drop=True)
LT.to_csv(f'{OUT}/tempflow_leadtime.csv', index=False)

fig, ax = plt.subplots(figsize=(6.3, 3.5))
best = LT[(LT.tempflow < LT.persistence) & (LT.tempflow < LT.climatology_pixel)].lead_days
for h in best:
    ax.axvspan(h - 0.5, h + 0.5, color='0.92', zorder=0)
ax.plot(LT.lead_days, LT.climatology_pixel, color='#d1711e', ls=':', lw=1.8, marker='^', ms=6, label='Climatology (seasonal cycle)')
ax.plot(LT.lead_days, LT.persistence, color='#3a6ea5', ls='--', lw=1.8, marker='s', ms=6, label='Persistence (last value)')
ax.plot(LT.lead_days, LT.tempflow, color='#111111', ls='-', lw=2.2, marker='o', ms=6.5, label='TempFlow (BiLSTM)')
ax.set_xlabel('Forecast lead time (days)'); ax.set_ylabel('Mean absolute error (K)')
ax.set_xticks(LT.lead_days.tolist())
ax.grid(True, axis='y', lw=0.4, color='0.85'); ax.set_axisbelow(True)
ax.legend(loc='best', frameon=True, framealpha=0.95, edgecolor='0.8', fontsize=8.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig(f'{OUT}/fig_tempflow_leadtime.png'); fig.savefig(f'{OUT}/fig_tempflow_leadtime.pdf'); plt.show()
print(LT.to_string(index=False))

## Forecasts for FusionFlow

In [ ]:
# next-day forecast for every pixel and day with a complete 30-day context (cloudy target days included)
m1 = MODELS[1]
FP, FD, FV, OP, OD, OV, PLAT, PLON = [], [], [], [], [], [], [], []
for k, (pid, p) in enumerate(df.groupby('pixel')):
    p = p.sort_values('Date'); n = len(p)
    PLAT.append(float(p['Latitude'].iloc[0])); PLON.append(float(p['Longitude'].iloc[0]))
    dnum = p['Date'].values.astype('datetime64[D]').astype('int64')
    rv = p['real'].values; o = ~np.isnan(rv)
    OP.append(np.full(o.sum(), k, 'int16')); OD.append(dnum[o].astype('int32')); OV.append(rv[o].astype('float32'))
    if n <= SEQ:
        continue
    ok = p['ctx'].notna().values
    Fs = np.full((n, len(FEATS)), np.nan, dtype='float32')
    Fs[ok] = sx.transform(p.loc[ok, FEATS])
    c = np.r_[0, np.cumsum(ok)]
    i = np.nonzero(((c[SEQ:] - c[:-SEQ]) == SEQ)[:n - SEQ])[0]
    if not len(i):
        continue
    W = np.lib.stride_tricks.sliding_window_view(Fs, SEQ, axis=0)[i].transpose(0, 2, 1)
    f = m1.predict(W, batch_size=4096, verbose=0).ravel() * YSCALE + YMEAN
    FP.append(np.full(len(i), k, 'int16')); FD.append(dnum[i + SEQ].astype('int32')); FV.append(f.astype('float32'))
FP, FD, FV = np.concatenate(FP), np.concatenate(FD), np.concatenate(FV)
OP, OD, OV = np.concatenate(OP), np.concatenate(OD), np.concatenate(OV)
CUT_DAY = int((CUT - pd.Timestamp('1970-01-01')).days)
np.savez_compressed(f'{OUT}/tempflow_forecasts_all.npz', pix_lat=np.array(PLAT), pix_lon=np.array(PLON),
                    fc_pix=FP, fc_day=FD, forecast=FV, obs_pix=OP, obs_day=OD, observed=OV, cut_day=CUT_DAY)
DAILY = (pd.DataFrame(dict(day=FD, fc=FV)).groupby('day').fc.agg(['size', 'mean'])
           .rename(columns={'size': 'n_forecast', 'mean': 'mean_forecast_K'})
           .join(pd.DataFrame(dict(day=OD, ob=OV)).groupby('day').ob.agg(['size', 'mean'])
                   .rename(columns={'size': 'n_observed', 'mean': 'mean_observed_K'}), how='outer').reset_index())
DAILY.insert(0, 'date', pd.to_datetime(DAILY.day, unit='D').dt.strftime('%Y-%m-%d'))
DAILY.round(3).to_csv(f'{OUT}/tempflow_daily_for_fusion.csv', index=False)
print(f'forecasts for {len(FV):,} pixel-days saved ({(FD >= CUT_DAY).mean():.0%} in the test period)')